# Alpha Signal Audit Viewer

Loads outputs from `tools/alpha_signal_audit.py` and shows:
- Per-signal behavior from backtest logs
- Single-signal backtest ranking


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

def _resolve_audit_root() -> Path:
    candidates = [
        Path('research/output/alpha_signal_audit'),
        Path('../research/output/alpha_signal_audit'),
    ]
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]

AUDIT_ROOT = _resolve_audit_root()
run_dirs = []
if AUDIT_ROOT.exists():
    run_dirs = sorted([d for d in AUDIT_ROOT.iterdir() if d.is_dir() and (d / 'manifest.json').exists()], key=lambda d: d.stat().st_mtime, reverse=True)
assert run_dirs, f'No audit runs found under {AUDIT_ROOT.resolve()}'

top = run_dirs[:5]
display(pd.DataFrame([
    {'rank': i + 1, 'run_dir': d.name, 'modified_at': pd.to_datetime(d.stat().st_mtime, unit='s')}
    for i, d in enumerate(top)
]))


In [ ]:
# Defaults to latest run. Uncomment to pin one.
RUN_DIR = run_dirs[0]
# RUN_DIR = AUDIT_ROOT / 'single_signal_2024_2026'

manifest = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
print('RUN_DIR:', RUN_DIR)
display(pd.DataFrame({'key': list(manifest.keys()), 'value': [manifest[k] for k in manifest.keys()]}))


In [ ]:
log_csv = RUN_DIR / 'signal_behavior_from_log.csv'
single_csv = RUN_DIR / 'single_signal_backtest_summary.csv'

if not log_csv.exists():
    cand = list((RUN_DIR / 'single_signal_runs').glob('*/../signal_behavior_from_log.csv'))

if log_csv.exists():
    log_df = pd.read_csv(log_csv)
    display(log_df.head(20))
else:
    log_df = pd.DataFrame()
    print('No signal_behavior_from_log.csv in', RUN_DIR)

if single_csv.exists():
    single_df = pd.read_csv(single_csv)
    display(single_df.head(20))
else:
    single_df = pd.DataFrame()
    print('No single_signal_backtest_summary.csv in', RUN_DIR)


In [ ]:
if not log_df.empty:
    view = log_df.sort_values('weighted_ic_proxy', ascending=False).head(15).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(view['signal'], view['weighted_ic_proxy'])
    ax.set_title('Per-Signal Weighted IC Proxy (Top 15)')
    ax.set_xlabel('weighted_ic_proxy')
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 6))
    view2 = log_df.sort_values('signal_ic_ir', ascending=False).head(15).iloc[::-1]
    ax.barh(view2['signal'], view2['signal_ic_ir'])
    ax.set_title('Per-Signal IC IR (Top 15)')
    ax.set_xlabel('signal_ic_ir')
    plt.tight_layout()
    plt.show()


In [ ]:
if not single_df.empty:
    cols = ['signal', 'type', 'sharpe', 'total_return', 'realized_active_information_ratio', 'h1_average_ic', 'h2_average_ic', 'h4_average_ic', 'average_executed_turnover']
    cols = [c for c in cols if c in single_df.columns]
    display(single_df[cols].sort_values('sharpe', ascending=False))

    plot_df = single_df.sort_values('sharpe', ascending=False).head(15).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(plot_df['signal'], plot_df['sharpe'])
    ax.set_title('Single-Signal Sharpe Ranking')
    ax.set_xlabel('sharpe')
    plt.tight_layout()
    plt.show()


## How To Use

1. Run `tools/alpha_signal_audit.py` in `log`, `single`, or `both` mode.
2. Open this notebook and pick the target `RUN_DIR`.
3. Use `signal_behavior_from_log.csv` for in-production behavior and `single_signal_backtest_summary.csv` for stand-alone strength.
